### AIRCRAFT LANDING SCHDULING

##### *Problem Statement*

The first task involves scheduling aircraft landings on a single runway. 
The objective is to *minimize the total penalty cost for early and late landings* ensuring aircraft land within their specified time windows and *required minimum separation times between 2 landings are maintained*.

##### *Data*
The data consists of the following tables: <br>
**Table 1**: Aircraft Time Windows and Penalty Costs <br>
**Table 2**: Minimum Separation Times 

##### *Model*

1. **Variables**: <br>
$t_i$ = landing time of aircraft $i$ <br>
$e_i$ = early landing penalty of aircraft $i$ <br>
$l_i$ = late landing penalty of aircraft $i$ <br>

2. **Objective Function**: <br>
Minimize the total penalty cost for early and late landings <br>
$Minimize \sum_{i} (penalty\_early_i . e_i + penalty\_late_i . l_i)$ 

3. **Constraints**: <br>
- Aircraft land within their specified time windows <br>
$t_i \geq Earliest\_landing_i$ <br>
$t_i \leq Latest\_landing_i$ <br>

- Penalty Calculation <br>
$e_i \geq max(0, Estimated_i - t_i)$ <br>
$l_i \geq max(0, t_i - Estimated_i)$

- Required minimum separation times between 2 landings are maintained <br>
$t_j \geq t_i + Separation_{i,j}\; i \not= j$




In [2]:
import gurobipy as gp
from gurobipy import GRB

# Data
aircraft = range(10)
earliest = [129, 195, 89, 96, 110, 120, 124, 126, 135, 160]
estimated = [155, 258, 96, 106, 123, 135, 138, 140, 150, 180]
latest = [559, 744, 510, 521, 555, 576, 577, 573, 591, 657]
penalty_early = [10, 10, 30, 30, 30, 30, 30, 30, 30, 30]
penalty_late = [10, 10, 30, 30, 30, 30, 30, 30, 30, 30]
separation = [
    [0, 3, 15, 15, 15, 15, 15, 15, 15, 15],
    [3, 0, 15, 15, 15, 15, 15, 15, 15, 15],
    [15, 15, 0, 8, 8, 8, 8, 8, 8, 8],
    [15, 15, 8, 0, 8, 8, 8, 8, 8, 8],
    [15, 15, 8, 8, 0, 8, 8, 8, 8, 8],
    [15, 15, 8, 8, 8, 0, 8, 8, 8, 8],
    [15, 15, 8, 8, 8, 8, 0, 8, 8, 8],
    [15, 15, 8, 8, 8, 8, 8, 0, 8, 8],
    [15, 15, 8, 8, 8, 8, 8, 8, 0, 8],
    [15, 15, 8, 8, 8, 8, 8, 8, 8, 0]
]

# Model
model = gp.Model("Aircraft_Landing")

# Variables
t = model.addVars(aircraft, vtype=GRB.CONTINUOUS, name="t")
e = model.addVars(aircraft, vtype=GRB.CONTINUOUS, name="e")
l = model.addVars(aircraft, vtype=GRB.CONTINUOUS, name="l")

# Objective
model.setObjective(gp.quicksum(penalty_early[i] * e[i] + penalty_late[i] * l[i] for i in aircraft), GRB.MINIMIZE)

# Constraints
for i in aircraft:
    model.addConstr(t[i] >= earliest[i])  # Landing time should be greater than or equal to the earliest time
    model.addConstr(t[i] <= latest[i])    # Landing time should be less than or equal to the latest time
    model.addConstr(e[i] >= estimated[i] - t[i])  # Early penalty calculation
    model.addConstr(l[i] >= t[i] - estimated[i])  # Late penalty calculation

# Separation time constraints
for i in aircraft:
    for j in aircraft:
        if i != j:
            model.addConstr(t[j] >= t[i] + separation[i][j])  # Minimum separation time between landings

# Optimize
model.optimize()

# Check if optimization was successful
if model.status == GRB.OPTIMAL:
    for i in aircraft:
        print(f"Aircraft {i+1}: Landing time = {t[i].X}, Early penalty = {e[i].X}, Late penalty = {l[i].X}")
else:
    print("Optimization was not successful.")
    if model.status == GRB.INFEASIBLE:
        print("Model is infeasible.")
    elif model.status == GRB.UNBOUNDED:
        print("Model is unbounded.")
    else:
        print(f"Optimization ended with status {model.status}")


Gurobi Optimizer version 11.0.2 build v11.0.2rc0 (win64 - Windows 10.0 (19045.2))

CPU model: Intel(R) Core(TM) i7-8550U CPU @ 1.80GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 130 rows, 30 columns and 240 nonzeros
Model fingerprint: 0xb78e074a
Coefficient statistics:


  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+01, 3e+01]
  Bounds range     [0e+00, 0e+00]
  RHS range        [3e+00, 7e+02]
Presolve removed 30 rows and 0 columns
Presolve time: 0.01s

Solved in 0 iterations and 0.02 seconds (0.00 work units)
Infeasible model
Optimization was not successful.
Model is infeasible.


In [ ]:
import gurobipy as gp
from gurobipy import GRB

# Data
aircraft = range(10)
earliest = [129, 195, 89, 96, 110, 120, 124, 126, 135, 160]
estimated = [155, 258, 96, 106, 123, 135, 138, 140, 150, 180]
latest = [559, 744, 510, 521, 555, 576, 577, 573, 591, 657]
penalty_early = [10, 10, 30, 30, 30, 30, 30, 30, 30, 30]
penalty_late = [10, 10, 30, 30, 30, 30, 30, 30, 30, 30]
separation = [
    [0, 3, 15, 15, 15, 15, 15, 15, 15, 15],
    [3, 0, 15, 15, 15, 15, 15, 15, 15, 15],
    [15, 15, 0, 8, 8, 8, 8, 8, 8, 8],
    [15, 15, 8, 0, 8, 8, 8, 8, 8, 8],
    [15, 15, 8, 8, 0, 8, 8, 8, 8, 8],
    [15, 15, 8, 8, 8, 0, 8, 8, 8, 8],
    [15, 15, 8, 8, 8, 8, 0, 8, 8, 8],
    [15, 15, 8, 8, 8, 8, 8, 0, 8, 8],
    [15, 15, 8, 8, 8, 8, 8, 8, 0, 8],
    [15, 15, 8, 8, 8, 8, 8, 8, 8, 0]
]

# Model
model = gp.Model("Aircraft_Landing")

# Variables
t = model.addVars(aircraft, vtype=GRB.CONTINUOUS, name="t")
e = model.addVars(aircraft, vtype=GRB.CONTINUOUS, name="e")
l = model.addVars(aircraft, vtype=GRB.CONTINUOUS, name="l")

# Objective
model.setObjective(gp.quicksum(penalty_early[i] * e[i] + penalty_late[i] * l[i] for i in aircraft), GRB.MINIMIZE)

# Constraints
for i in aircraft:
    model.addConstr(t[i] >= earliest[i], name=f"earliest_{i}")  # Landing time should be greater than or equal to the earliest time
    model.addConstr(t[i] <= latest[i], name=f"latest_{i}")    # Landing time should be less than or equal to the latest time
    model.addConstr(e[i] >= estimated[i] - t[i], name=f"early_penalty_{i}")  # Early penalty calculation
    model.addConstr(l[i] >= t[i] - estimated[i], name=f"late_penalty_{i}")  # Late penalty calculation

# Separation time constraints
for i in aircraft:
    for j in aircraft:
        if i != j:
            model.addConstr(t[j] >= t[i] + separation[i][j], name=f"separation_{i}_{j}")  # Minimum separation time between landings

# Optimize
model.optimize()

# Check if optimization was successful
if model.status == GRB.OPTIMAL:
    for i in aircraft:
        print(f"Aircraft {i+1}: Landing time = {t[i].X}, Early penalty = {e[i].X}, Late penalty = {l[i].X}")
else:
    print("Optimization was not successful.")
    if model.status == GRB.INFEASIBLE:
        print("Model is infeasible.")
        # Compute IIS
        model.computeIIS()
        model.write("model.ilp")
        for constr in model.getConstrs():
            if constr.IISConstr:
                print(f"Infeasible constraint: {constr.ConstrName}")
    elif model.status == GRB.UNBOUNDED:
        print("Model is unbounded.")
    else:
        print(f"Optimization ended with status {model.status}")
